# EUI Prediction Pipeline — Google Colab Runner

**Paper:** Explainable Deep Learning for Equity Market Uncertainty Prediction (JAIS Revision)

**Instructions:**
1. Runtime → Change runtime type → **A100 GPU** (or T4)
2. Mount your Google Drive containing the repo and data
3. Run all cells in order
4. Outputs saved to `outputs/final_report/` in your Drive

**Estimated runtime (full mode, A100):** ~3-6 hours depending on dataset sizes

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set your repo path on Drive (update if different)
import os
REPO_PATH = '/content/drive/MyDrive/market-uncertainty-prediction'  # EDIT THIS
DATA_PATH = '/content/drive/MyDrive/eui-data'  # EDIT THIS — where raw data CSVs are stored

if not os.path.exists(REPO_PATH):
    # Clone repo to Drive if not present
    !git clone https://github.com/YOUR_USERNAME/market-uncertainty-prediction.git "{REPO_PATH}"

os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# Cell 3: Install dependencies
!pip install -q neuralforecast shap scikit-learn gdown pyyaml jinja2 beautifulsoup4 lxml pyarrow fastparquet nltk tqdm scipy openpyxl

# Install GDCM
if not os.path.exists('gdcm'):
    !git clone https://github.com/cygit/gdcm.git gdcm
!pip install -q -e gdcm/src

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

print('Dependencies installed.')

In [ ]:
# Cell 4: Copy data files from Drive to repo data/ directory
import shutil

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/eui', exist_ok=True)

# Copy StackExchange and Bogleheads if available on Drive
for fname in ['StackExchange_.csv', 'bogleheads_.csv']:
    src = os.path.join(DATA_PATH, fname)
    dst = os.path.join('data', fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'Copied {fname}')
    elif os.path.exists(dst):
        print(f'Already present: {dst}')
    else:
        print(f'WARNING: {src} not found on Drive. Run 00_download_data.py to fetch.')

!ls data/

In [ ]:
# Cell 5: Run full pipeline
# Adjust DATASET and MODE as needed
DATASET = 'all'   # all | reddit | stackexchange | bogleheads
MODE = 'full'     # full | smoke

!bash run_pipeline.sh {DATASET} {MODE}

In [ ]:
# Cell 6: Copy outputs back to Google Drive
import shutil

drive_outputs = os.path.join(DATA_PATH, 'outputs')
os.makedirs(drive_outputs, exist_ok=True)

if os.path.exists('outputs'):
    shutil.copytree('outputs', drive_outputs, dirs_exist_ok=True)
    print(f'Outputs copied to Drive: {drive_outputs}')

!ls outputs/final_report/

In [ ]:
# Cell 7: Display reports inline
from IPython.display import HTML, display
import glob

report_files = sorted(glob.glob('outputs/final_report/*.html'))
print(f'Reports generated: {report_files}')

for rpt in report_files[:1]:  # Show first report inline
    with open(rpt) as f:
        content = f.read()
    display(HTML(f'<h3>{rpt}</h3>' + content[:50000]))  # Truncate for display